# Sales Forecasting MLOps - Data Registration

## Business Context

A sales forecast predicts future sales revenue based on historical data, industry trends, and the status of the current sales pipeline. Businesses use sales forecasts to estimate weekly, monthly, quarterly, and annual sales totals. Accurate forecasting adds value across an organization by helping different business verticals plan their future course of action.

Forecasting helps an organization plan sales operations by region and provides valuable insights to the supply chain team regarding the procurement of goods and materials. An accurate sales forecasting process improves decision-making about the future, reduces sales pipeline and forecast risks, reduces the time spent planning territory coverage, and establishes benchmarks that can be used to assess future trends.

## Objective

As an MLOps Engineer, the task is to build an automated MLOps pipeline with CI/CD that delivers accurate and reliable sales forecasts. The solution will use historical sales data, industry trends, and current pipeline status to predict weekly, monthly, quarterly, and annual revenue.

Automating data ingestion, preprocessing, model training, evaluation, and deployment will provide scalability, consistency, and minimal manual intervention. CI/CD integration will continuously validate changes, update forecasts, and deploy approved models. This enables business verticals to plan regional sales operations, optimize supply chain procurement, reduce sales pipeline risk, establish benchmarks for trend analysis, improve operational efficiency, and support business growth.

## Data Dictionary

| Attribute | Description |
|---|---|
| `Product_Id` | Unique identifier for each product, beginning with two letters followed by a number. |
| `Product_Weight` | Weight of the product. |
| `Product_Sugar_Content` | Sugar-content category, such as Low Sugar, Regular, or No Sugar. |
| `Product_Allocated_Area` | Ratio of the product's allocated display area to the total display area of all products in a store. |
| `Product_Type` | Broad product category, such as meat, snack foods, hard drinks, dairy, canned goods, soft drinks, health and hygiene, baking goods, bread, breakfast, frozen foods, fruits and vegetables, household, seafood, starchy foods, or others. |
| `Product_MRP` | Maximum retail price of the product. |
| `Store_Id` | Unique identifier for each store. |
| `Store_Establishment_Year` | Year in which the store was established. |
| `Store_Size` | Store size based on square footage, categorized as High, Medium, or Low. |
| `Store_Location_City_Type` | City tier in which the store is located: Tier 1, Tier 2, or Tier 3. Tier 1 cities generally have a comparatively higher standard of living. |
| `Store_Type` | Store category, such as Departmental Store, Supermarket Type 1, Supermarket Type 2, or Food Mart. |
| `Product_Store_Sales_Total` | **Target variable:** total revenue generated by a product in a particular store. |

## Data Registration Workflow

1. Create or use the project master folder, `mlops_sales_forecasting`.
2. Create a `data` subfolder inside the master folder.
3. Place `SuperKart.csv` inside the `data` folder.
4. Install the required Python dependencies.
5. Authenticate securely with Hugging Face.
6. Load and validate the local data and schema.
7. Create a Hugging Face dataset repository.
8. Upload the dataset as a `train` split.
9. Load the remote dataset and verify the registration.

In [1]:
import os
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd

if "google.colab" in sys.modules:
    PROJECT_ROOT = Path("/content/mlops_sales_forecasting")
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name.lower() == "data":
        PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = DATA_DIR / "SuperKart.csv"

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Data folder: {DATA_DIR.resolve()}")
print(f"Expected CSV: {CSV_PATH.resolve()}")

Project root: C:\Users\arvnarayanan\Documents\mlops_sales_forecasting
Data folder: C:\Users\arvnarayanan\Documents\mlops_sales_forecasting\data
Expected CSV: C:\Users\arvnarayanan\Documents\mlops_sales_forecasting\data\SuperKart.csv


### Place the source CSV in the data folder

Copy or upload `SuperKart.csv` to the expected path printed above. The notebook never overwrites the source file.

In Google Colab, an optional upload flow is:

```python
from google.colab import files
uploaded = files.upload()
# Move the uploaded SuperKart.csv to CSV_PATH after reviewing the filename.
```

In [2]:
REQUIRED_COLUMNS = [
    "Product_Id",
    "Product_Weight",
    "Product_Sugar_Content",
    "Product_Allocated_Area",
    "Product_Type",
    "Product_MRP",
    "Store_Id",
    "Store_Establishment_Year",
    "Store_Size",
    "Store_Location_City_Type",
    "Store_Type",
    "Product_Store_Sales_Total",
]

if not CSV_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {CSV_PATH.resolve()}. "
        "Copy SuperKart.csv into the data folder and rerun this cell."
    )

df = pd.read_csv(CSV_PATH)
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
if missing_columns:
    raise ValueError(f"The CSV is missing required columns: {missing_columns}")

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Duplicate rows: {df.duplicated().sum():,}")
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing_values": df.isna().sum()}))
display(df.head())

Shape: 8,763 rows x 12 columns
Duplicate rows: 0


,dtype,missing_values
Product_Id,str,0
Product_Weight,float64,0
Product_Sugar_Content,str,0
Product_Allocated_Area,float64,0
Product_Type,str,0
Product_MRP,float64,0
Store_Id,str,0
Store_Establishment_Year,int64,0
Store_Size,str,0
Store_Location_City_Type,str,0


,Product_Id,Product_Weight,Product_Sugar_Content,Product_Allocated_Area,Product_Type,Product_MRP,Store_Id,Store_Establishment_Year,Store_Size,Store_Location_City_Type,Store_Type,Product_Store_Sales_Total
0,FD6114,12.66,Low Sugar,0.027,Frozen Foods,117.08,OUT004,2009,Medium,Tier 2,Supermarket Type2,2842.40
1,FD7839,16.54,Low Sugar,0.144,Dairy,171.43,OUT003,1999,Medium,Tier 1,Departmental Store,4830.02
2,FD5075,14.28,Regular,0.031,Canned,162.08,OUT001,1987,High,Tier 2,Supermarket Type1,4130.16
3,FD8233,12.10,Low Sugar,0.112,Baking Goods,186.31,OUT001,1987,High,Tier 2,Supermarket Type1,4132.18
4,NC1180,9.57,No Sugar,0.010,Health and Hygiene,123.67,OUT002,1998,Small,Tier 3,Food Mart,2279.36


## Install Dependencies

The next cell installs only packages that are not already available in the active notebook environment.

In [3]:
import importlib.util
import subprocess

required_packages = {
    "huggingface_hub": "huggingface_hub>=0.20.0",
    "datasets": "datasets>=2.16.0",
}
missing_packages = [
    requirement
    for module, requirement in required_packages.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("All Hugging Face dependencies are already installed.")

All Hugging Face dependencies are already installed.


## Secure Hugging Face Authentication

1. Create a Hugging Face account if needed.
2. Open [Hugging Face access tokens](https://huggingface.co/settings/tokens).
3. Create a token with **write** permission so the notebook can create and update a dataset repository.
4. Never paste a token into a notebook cell, commit it to source control, or print it.

This notebook supports either an `HF_TOKEN` environment variable or a hidden `getpass` prompt. Set `HF_USERNAME` to the Hugging Face account or organization and optionally set `HF_DATASET_NAME`; the default dataset name is `superkart-sales-forecasting`.

In [4]:
from huggingface_hub import login

HF_TOKEN = os.getenv("HF_TOKEN", "").strip()
HF_USERNAME = os.getenv("HF_USERNAME", "").strip()
HF_DATASET_NAME = os.getenv("HF_DATASET_NAME", "superkart-sales-forecasting").strip()

if not HF_TOKEN:
    HF_TOKEN = getpass("Enter your Hugging Face write token (input is hidden): ").strip()
if not HF_USERNAME:
    HF_USERNAME = input("Enter your Hugging Face username or organization: ").strip()

if not HF_TOKEN:
    raise ValueError("A Hugging Face write token is required.")
if not HF_USERNAME:
    raise ValueError("HF_USERNAME is required.")
if not HF_DATASET_NAME:
    raise ValueError("HF_DATASET_NAME cannot be empty.")

REPO_ID = f"{HF_USERNAME}/{HF_DATASET_NAME}"
login(token=HF_TOKEN, add_to_git_credential=False)
print(f"Authenticated for dataset repository: {REPO_ID}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Authenticated for dataset repository: arvind2221994/superkart-sales-forecasting


## Register the Dataset

The cell below creates a **public** dataset repository and uploads the validated data as its `train` split. Set `private=True` in `create_repo` if access must be restricted. Rerunning the cell updates the same repository instead of creating a duplicate.

In [5]:
from datasets import Dataset, DatasetDict
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(
    repo_id=REPO_ID,
    repo_type="dataset",
    private=False,
    exist_ok=True,
)

dataset = Dataset.from_pandas(df, preserve_index=False)
dataset_dict = DatasetDict({"train": dataset})
dataset_dict.push_to_hub(repo_id=REPO_ID, token=HF_TOKEN)

DATASET_URL = f"https://huggingface.co/datasets/{REPO_ID}"
print(f"Dataset registered successfully: {DATASET_URL}")

c:\Users\arvnarayanan\Documents\mlops_sales_forecasting\.venv-x64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 36.74ba/s]
Upload 0 LFS files: 0it [00:00, ?it/s]
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.58 shards/s]


Dataset registered successfully: https://huggingface.co/datasets/arvind2221994/superkart-sales-forecasting


## Verify Registration

Reload the `train` split from Hugging Face and compare its schema and row count with the local validated data.

In [6]:
from datasets import load_dataset

remote_dataset = load_dataset(REPO_ID, split="train", token=HF_TOKEN)
remote_columns = remote_dataset.column_names

print(f"Remote rows: {len(remote_dataset):,}")
print(f"Remote columns ({len(remote_columns)}): {remote_columns}")

missing_remote_columns = sorted(set(REQUIRED_COLUMNS) - set(remote_columns))
assert not missing_remote_columns, f"Remote dataset is missing: {missing_remote_columns}"
assert len(remote_dataset) == len(df), (
    f"Row-count mismatch: remote={len(remote_dataset):,}, local={len(df):,}"
)

print("Remote schema and row count match the local dataset.")
display(remote_dataset.select(range(min(5, len(remote_dataset)))).to_pandas())

c:\Users\arvnarayanan\Documents\mlops_sales_forecasting\.venv-x64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\arvnarayanan\.cache\huggingface\hub\datasets--arvind2221994--superkart-sales-forecasting. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\arvnarayanan\Documents\mlops_sales_forecast

Remote rows: 8,763
Remote columns (12): ['Product_Id', 'Product_Weight', 'Product_Sugar_Content', 'Product_Allocated_Area', 'Product_Type', 'Product_MRP', 'Store_Id', 'Store_Establishment_Year', 'Store_Size', 'Store_Location_City_Type', 'Store_Type', 'Product_Store_Sales_Total']
Remote schema and row count match the local dataset.


,Product_Id,Product_Weight,Product_Sugar_Content,Product_Allocated_Area,Product_Type,Product_MRP,Store_Id,Store_Establishment_Year,Store_Size,Store_Location_City_Type,Store_Type,Product_Store_Sales_Total
0,FD6114,12.66,Low Sugar,0.027,Frozen Foods,117.08,OUT004,2009,Medium,Tier 2,Supermarket Type2,2842.40
1,FD7839,16.54,Low Sugar,0.144,Dairy,171.43,OUT003,1999,Medium,Tier 1,Departmental Store,4830.02
2,FD5075,14.28,Regular,0.031,Canned,162.08,OUT001,1987,High,Tier 2,Supermarket Type1,4130.16
3,FD8233,12.10,Low Sugar,0.112,Baking Goods,186.31,OUT001,1987,High,Tier 2,Supermarket Type1,4132.18
4,NC1180,9.57,No Sugar,0.010,Health and Hygiene,123.67,OUT002,1998,Small,Tier 3,Food Mart,2279.36


## Data Preparation

After registering and verifying the raw dataset, prepare model-ready training and testing splits directly from the Hugging Face dataset repository. This workflow follows `prepare_data.py` in this repository:

1. **Load from Hugging Face:** Read the current `train` split from `REPO_ID` instead of relying on the local raw CSV. This makes the registered dataset the shared source of truth.
2. **Clean the data:** Standardize inconsistent `Product_Sugar_Content` values (`low sugar`, `reg`, and `no sugar`), fill missing `Product_Weight` values with the median, fill missing `Store_Size` values with `Unknown`, and remove the unnecessary `__index_level_0__` column if it exists.
3. **Create reproducible splits:** Split the cleaned data into 80% training and 20% testing data with `random_state=42`.
4. **Save locally:** Write the prepared files to `data/processed/train.csv` and `data/processed/test.csv` without DataFrame index columns.
5. **Update Hugging Face:** Upload a `DatasetDict` containing both `train` and `test` splits to the same dataset repository. This updates the repository used by the downstream training pipeline.

> The preparation code is intentionally split into two cells. Review the cleaning and split summaries before running the upload cell, because uploading replaces the repository's split configuration with the prepared `train` and `test` splits.

In [7]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# Load the registered raw data from the Hugging Face dataset repository.
raw_dataset = load_dataset(REPO_ID, split="train", token=HF_TOKEN)
prepared_df = raw_dataset.to_pandas()

missing_columns = sorted(set(REQUIRED_COLUMNS) - set(prepared_df.columns))
if missing_columns:
    raise ValueError(f"The Hugging Face dataset is missing columns: {missing_columns}")

# Apply the same cleaning rules used by prepare_data.py.
prepared_df["Product_Sugar_Content"] = prepared_df["Product_Sugar_Content"].replace(
    {"low sugar": "Low Sugar", "reg": "Regular", "no sugar": "No Sugar"}
)
prepared_df["Product_Weight"] = prepared_df["Product_Weight"].fillna(
    prepared_df["Product_Weight"].median()
)
prepared_df["Store_Size"] = prepared_df["Store_Size"].fillna("Unknown")
prepared_df = prepared_df.drop(columns=["__index_level_0__"], errors="ignore")

train_df, test_df = train_test_split(
    prepared_df,
    test_size=0.2,
    random_state=42,
)

PROCESSED_DATA_DIR = DATA_DIR / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_PATH = PROCESSED_DATA_DIR / "train.csv"
TEST_PATH = PROCESSED_DATA_DIR / "test.csv"

train_df.to_csv(TRAIN_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)

print(f"Source rows: {len(prepared_df):,}")
print(f"Training rows: {len(train_df):,} ({len(train_df) / len(prepared_df):.1%})")
print(f"Testing rows: {len(test_df):,} ({len(test_df) / len(prepared_df):.1%})")
print(f"Training data saved to: {TRAIN_PATH.resolve()}")
print(f"Testing data saved to: {TEST_PATH.resolve()}")
display(prepared_df.isna().sum().rename("missing_values").to_frame())

Source rows: 8,763
Training rows: 7,010 (80.0%)
Testing rows: 1,753 (20.0%)
Training data saved to: C:\Users\arvnarayanan\Documents\mlops_sales_forecasting\data\processed\train.csv
Testing data saved to: C:\Users\arvnarayanan\Documents\mlops_sales_forecasting\data\processed\test.csv


,missing_values
Product_Id,0
Product_Weight,0
Product_Sugar_Content,0
Product_Allocated_Area,0
Product_Type,0
Product_MRP,0
Store_Id,0
Store_Establishment_Year,0
Store_Size,0
Store_Location_City_Type,0


### Upload the Prepared Splits

Run the following cell after reviewing the preparation output. It uploads both splits to the existing Hugging Face dataset repository and then reloads them to verify their row counts.

In [8]:
from datasets import Dataset, DatasetDict, load_dataset

processed_dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df, preserve_index=False),
        "test": Dataset.from_pandas(test_df, preserve_index=False),
    }
)
processed_dataset.push_to_hub(
    repo_id=REPO_ID,
    token=HF_TOKEN,
    private=False,
)

uploaded_dataset = load_dataset(REPO_ID, token=HF_TOKEN)
assert len(uploaded_dataset["train"]) == len(train_df)
assert len(uploaded_dataset["test"]) == len(test_df)

print(f"Prepared data uploaded to: https://huggingface.co/datasets/{REPO_ID}")
print(uploaded_dataset)

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 17.23ba/s]
tmp70ommu_o.parquet: 100%|██████████| 173k/173k [00:01<00:00, 127kB/s] 
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.30s/ shards]
Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 33.74ba/s]
tmpg940uqi3.parquet: 100%|██████████| 51.6k/51.6k [00:00<00:00, 99.0kB/s]
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.42s/ shards]
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not ins

Prepared data uploaded to: https://huggingface.co/datasets/arvind2221994/superkart-sales-forecasting
DatasetDict({
    train: Dataset({
        features: ['Product_Id', 'Product_Weight', 'Product_Sugar_Content', 'Product_Allocated_Area', 'Product_Type', 'Product_MRP', 'Store_Id', 'Store_Establishment_Year', 'Store_Size', 'Store_Location_City_Type', 'Store_Type', 'Product_Store_Sales_Total'],
        num_rows: 7010
    })
    test: Dataset({
        features: ['Product_Id', 'Product_Weight', 'Product_Sugar_Content', 'Product_Allocated_Area', 'Product_Type', 'Product_MRP', 'Store_Id', 'Store_Establishment_Year', 'Store_Size', 'Store_Location_City_Type', 'Store_Type', 'Product_Store_Sales_Total'],
        num_rows: 1753
    })
})


## Model Training and Registration

The repository trains six regression algorithms using the prepared Hugging Face `train` and `test` splits. Every training workflow follows the same lifecycle:

1. **Load data:** Download the prepared `train` and `test` splits from the Hugging Face dataset repository.
2. **Build features:** Remove `Product_Id` and `Store_Id`, because they are identifiers rather than generalizable predictors, and separate `Product_Store_Sales_Total` as the target.
3. **Preprocess:** Impute numeric columns with their median, impute categorical columns with their most frequent value, and one-hot encode categories while ignoring previously unseen values.
4. **Define model and parameters:** Create a complete scikit-learn pipeline and a model-specific hyperparameter grid.
5. **Tune:** Run five-fold `GridSearchCV`, selecting the configuration with the lowest cross-validation RMSE.
6. **Evaluate:** Measure RMSE, MAE, and R2 on the held-out test split. RMSE is the primary selection metric; lower RMSE and MAE are better, while higher R2 is better.
7. **Register:** Save the fitted pipeline as a Joblib artifact and upload `model.joblib` to its model repository in the Hugging Face Model Hub.

### Repository Folder Structure

```text
mlops_sales_forecasting/
|-- data/
|   |-- SuperKart.csv
|   `-- processed/
|       |-- train.csv
|       `-- test.csv
|-- scripts/
|   |-- train_decision_tree.py
|   |-- train_bagging.py
|   |-- train_random_forest.py
|   |-- train_adaboost.py
|   |-- train_gradient_boosting.py
|   |-- train_xgboost.py
|   `-- compare_models.py
|-- models/
|   |-- decision_tree_model.joblib
|   |-- bagging_model.joblib
|   |-- random_forest_model.joblib
|   |-- adaboost_model.joblib
|   |-- gradient_boosting_model.joblib
|   `-- xgboost_model.joblib
|-- reports/
|   |-- model_comparison.csv
|   `-- model_training_summary.md
|-- utils.py
|-- prepare_data.py
`-- requirements.txt
```

> The following training cells can take several minutes because each performs five-fold grid search. Each successful cell also updates a Hugging Face model repository.

### Training Prerequisites

Install only missing training dependencies. The notebook reuses the previously authenticated `HF_TOKEN`, `HF_USERNAME`, and `REPO_ID` values.

In [9]:
training_packages = {
    "sklearn": "scikit-learn>=1.3.0",
    "joblib": "joblib",
    "xgboost": "xgboost>=2.0.0",
}
missing_training_packages = [
    requirement
    for module, requirement in training_packages.items()
    if importlib.util.find_spec(module) is None
]

if missing_training_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing_training_packages]
    )
else:
    print("All model-training dependencies are already installed.")

All model-training dependencies are already installed.


### Shared Training Workflow

The repository repeats the same loading, preprocessing, evaluation, saving, and registration logic in every training script. The notebook factors that behavior into reusable functions so each model cell needs to specify only its estimator, parameter grid, artifact name, and Hugging Face repository name.

In [10]:
import json

import joblib
import numpy as np
from datasets import load_dataset
from huggingface_hub import HfApi
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

TARGET_COLUMN = "Product_Store_Sales_Total"
IDENTIFIER_COLUMNS = ["Product_Id", "Store_Id"]
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

train_data = load_dataset(REPO_ID, split="train", token=HF_TOKEN).to_pandas()
test_data = load_dataset(REPO_ID, split="test", token=HF_TOKEN).to_pandas()

for split_name, split_df in {"train": train_data, "test": test_data}.items():
    if TARGET_COLUMN not in split_df.columns:
        raise ValueError(f"The {split_name} split is missing {TARGET_COLUMN}.")

feature_drop_columns = [*IDENTIFIER_COLUMNS, TARGET_COLUMN]
X_train = train_data.drop(
    columns=[column for column in feature_drop_columns if column in train_data.columns]
)
y_train = train_data[TARGET_COLUMN]
X_test = test_data.drop(
    columns=[column for column in feature_drop_columns if column in test_data.columns]
)
y_test = test_data[TARGET_COLUMN]

def build_preprocessor(features):
    categorical_columns = features.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical_columns = features.select_dtypes(include=["number"]).columns.tolist()

    numeric_transformer = Pipeline(
        steps=[("imputer", SimpleImputer(strategy="median"))]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numerical_columns),
            ("cat", categorical_transformer, categorical_columns),
        ]
    )

def evaluate_regression(y_true, y_pred):
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }

def train_tune_evaluate_register(
    model_name,
    estimator,
    parameter_grid,
    artifact_name,
    model_repository_name,
):
    pipeline = Pipeline(
        [
            ("preprocessor", build_preprocessor(X_train)),
            ("regressor", estimator),
        ]
    )
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=parameter_grid,
        cv=5,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )
    print(f"Tuning {model_name}...")
    search.fit(X_train, y_train)

    best_model = search.best_estimator_
    predictions = best_model.predict(X_test)
    metrics = evaluate_regression(y_test, predictions)
    model_path = MODELS_DIR / artifact_name
    joblib.dump(best_model, model_path)

    model_repo_id = f"{HF_USERNAME}/{model_repository_name}"
    api = HfApi(token=HF_TOKEN)
    api.create_repo(
        repo_id=model_repo_id,
        repo_type="model",
        exist_ok=True,
    )
    api.upload_file(
        path_or_fileobj=str(model_path),
        path_in_repo="model.joblib",
        repo_id=model_repo_id,
        repo_type="model",
    )

    result = {
        "Model": model_name,
        **metrics,
        "CV_RMSE": float(-search.best_score_),
        "Best_Parameters": json.dumps(search.best_params_, sort_keys=True),
        "Artifact": str(model_path),
        "Model_Repository": model_repo_id,
    }
    print(f"Best parameters: {search.best_params_}")
    print(f"Test metrics: {metrics}")
    print(f"Registered at: https://huggingface.co/{model_repo_id}")
    return result

training_results = []
print(f"Training rows: {len(X_train):,}; testing rows: {len(X_test):,}")
print(f"Model features ({X_train.shape[1]}): {X_train.columns.tolist()}")

Training rows: 7,010; testing rows: 1,753
Model features (9): ['Product_Weight', 'Product_Sugar_Content', 'Product_Allocated_Area', 'Product_Type', 'Product_MRP', 'Store_Establishment_Year', 'Store_Size', 'Store_Location_City_Type', 'Store_Type']


### 1. Decision Tree Regressor

A decision tree creates interpretable rules by recursively splitting the feature space. The grid tunes tree depth and the minimum number of samples required to split a node. Restricting these values helps control overfitting.

| Parameter | Values |
|---|---|
| `max_depth` | 5, 10, 15, unlimited |
| `min_samples_split` | 2, 10, 20 |

In [11]:
from sklearn.tree import DecisionTreeRegressor

decision_tree_result = train_tune_evaluate_register(
    model_name="Decision Tree",
    estimator=DecisionTreeRegressor(random_state=42),
    parameter_grid={
        "regressor__max_depth": [5, 10, 15, None],
        "regressor__min_samples_split": [2, 10, 20],
    },
    artifact_name="decision_tree_model.joblib",
    model_repository_name="superkart-decision-tree",
)
training_results.append(decision_tree_result)

Tuning Decision Tree...


decision_tree_model.joblib: 100%|██████████| 42.6k/42.6k [00:00<00:00, 44.0kB/s]


Best parameters: {'regressor__max_depth': 10, 'regressor__min_samples_split': 20}
Test metrics: {'RMSE': 332.46566539524093, 'MAE': 149.61711187291598, 'R2': 0.9031275356623896}
Registered at: https://huggingface.co/arvind2221994/superkart-decision-tree


### 2. Bagging Regressor

Bagging fits multiple decision trees on sampled subsets and averages their predictions. This reduces the variance of a single decision tree. The grid tunes the number of trees and the fraction of training rows sampled by each estimator.

| Parameter | Values |
|---|---|
| `n_estimators` | 10, 30, 50 |
| `max_samples` | 0.7, 1.0 |

In [12]:
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor

bagging_result = train_tune_evaluate_register(
    model_name="Bagging",
    estimator=BaggingRegressor(
        estimator=DecisionTreeRegressor(),
        random_state=42,
    ),
    parameter_grid={
        "regressor__n_estimators": [10, 30, 50],
        "regressor__max_samples": [0.7, 1.0],
    },
    artifact_name="bagging_model.joblib",
    model_repository_name="superkart-bagging",
)
training_results.append(bagging_result)

Tuning Bagging...


bagging_model.joblib: 100%|██████████| 25.4M/25.4M [00:02<00:00, 8.83MB/s]


Best parameters: {'regressor__max_samples': 0.7, 'regressor__n_estimators': 50}
Test metrics: {'RMSE': 284.591376169618, 'MAE': 111.29783148887623, 'R2': 0.9290176755676917}
Registered at: https://huggingface.co/arvind2221994/superkart-bagging


### 3. Random Forest Regressor

Random forest adds random feature selection to bagged decision trees, producing a diverse ensemble that is typically robust on mixed tabular data. The grid tunes forest size, tree depth, and the minimum samples required for a split.

| Parameter | Values |
|---|---|
| `n_estimators` | 50, 100, 200 |
| `max_depth` | 8, 12, 16 |
| `min_samples_split` | 2, 5 |

In [13]:
from sklearn.ensemble import RandomForestRegressor

random_forest_result = train_tune_evaluate_register(
    model_name="Random Forest",
    estimator=RandomForestRegressor(random_state=42),
    parameter_grid={
        "regressor__n_estimators": [50, 100, 200],
        "regressor__max_depth": [8, 12, 16],
        "regressor__min_samples_split": [2, 5],
    },
    artifact_name="random_forest_model.joblib",
    model_repository_name="superkart-random-forest",
)
training_results.append(random_forest_result)

Tuning Random Forest...


random_forest_model.joblib: 100%|██████████| 19.9M/19.9M [00:04<00:00, 4.24MB/s]


Best parameters: {'regressor__max_depth': 12, 'regressor__min_samples_split': 5, 'regressor__n_estimators': 200}
Test metrics: {'RMSE': 278.25595323166397, 'MAE': 110.31528157283213, 'R2': 0.9321428406834187}
Registered at: https://huggingface.co/arvind2221994/superkart-random-forest


### 4. AdaBoost Regressor

AdaBoost trains weak learners sequentially and gives greater attention to observations with larger errors. The grid balances the number of boosting stages against the learning rate applied at each stage.

| Parameter | Values |
|---|---|
| `n_estimators` | 50, 100, 150 |
| `learning_rate` | 0.01, 0.1, 1.0 |

In [14]:
from sklearn.ensemble import AdaBoostRegressor

adaboost_result = train_tune_evaluate_register(
    model_name="AdaBoost",
    estimator=AdaBoostRegressor(random_state=42),
    parameter_grid={
        "regressor__n_estimators": [50, 100, 150],
        "regressor__learning_rate": [0.01, 0.1, 1.0],
    },
    artifact_name="adaboost_model.joblib",
    model_repository_name="superkart-adaboost",
)
training_results.append(adaboost_result)

Tuning AdaBoost...


adaboost_model.joblib: 100%|██████████| 75.3k/75.3k [00:01<00:00, 65.6kB/s]


Best parameters: {'regressor__learning_rate': 0.01, 'regressor__n_estimators': 50}
Test metrics: {'RMSE': 449.9570952383178, 'MAE': 346.4199787410213, 'R2': 0.8225610518314401}
Registered at: https://huggingface.co/arvind2221994/superkart-adaboost


### 5. Gradient Boosting Regressor

Gradient boosting builds trees sequentially, with each tree correcting residual errors from the existing ensemble. The grid tunes the number of boosting stages, shrinkage rate, and tree complexity.

| Parameter | Values |
|---|---|
| `n_estimators` | 100, 200 |
| `learning_rate` | 0.03, 0.1 |
| `max_depth` | 3, 5 |

In [15]:
from sklearn.ensemble import GradientBoostingRegressor

gradient_boosting_result = train_tune_evaluate_register(
    model_name="Gradient Boosting",
    estimator=GradientBoostingRegressor(random_state=42),
    parameter_grid={
        "regressor__n_estimators": [100, 200],
        "regressor__learning_rate": [0.03, 0.1],
        "regressor__max_depth": [3, 5],
    },
    artifact_name="gradient_boosting_model.joblib",
    model_repository_name="superkart-gradient-boosting",
)
training_results.append(gradient_boosting_result)

Tuning Gradient Boosting...


gradient_boosting_model.joblib: 100%|██████████| 899k/899k [00:01<00:00, 569kB/s] 


Best parameters: {'regressor__learning_rate': 0.03, 'regressor__max_depth': 5, 'regressor__n_estimators': 200}
Test metrics: {'RMSE': 284.0844120677817, 'MAE': 121.34584936852099, 'R2': 0.9292703426653441}
Registered at: https://huggingface.co/arvind2221994/superkart-gradient-boosting


### 6. XGBoost Regressor

XGBoost is an optimized gradient-boosted tree implementation with efficient parallel training and regularized tree construction. The grid tunes ensemble size, learning rate, depth, and row subsampling.

| Parameter | Values |
|---|---|
| `n_estimators` | 100, 200 |
| `learning_rate` | 0.03, 0.1 |
| `max_depth` | 3, 6 |
| `subsample` | 0.8, 1.0 |

In [16]:
from xgboost import XGBRegressor

xgboost_result = train_tune_evaluate_register(
    model_name="XGBoost",
    estimator=XGBRegressor(random_state=42, n_jobs=-1),
    parameter_grid={
        "regressor__n_estimators": [100, 200],
        "regressor__learning_rate": [0.03, 0.1],
        "regressor__max_depth": [3, 6],
        "regressor__subsample": [0.8, 1.0],
    },
    artifact_name="xgboost_model.joblib",
    model_repository_name="superkart-xgboost",
)
training_results.append(xgboost_result)

Tuning XGBoost...


xgboost_model.joblib: 100%|██████████| 836k/836k [00:01<00:00, 478kB/s] 


Best parameters: {'regressor__learning_rate': 0.03, 'regressor__max_depth': 6, 'regressor__n_estimators': 200, 'regressor__subsample': 0.8}
Test metrics: {'RMSE': 285.9940687185443, 'MAE': 116.40167961786491, 'R2': 0.9283162365426478}
Registered at: https://huggingface.co/arvind2221994/superkart-xgboost


## Model Comparison and Best-Model Registration

The repository's `scripts/compare_models.py` provides a separate, reproducible selection stage after individual model training:

1. Map each model name to its local Joblib artifact in `models/`.
2. Load the prepared Hugging Face test split and remove identifier and target columns from the features.
3. Load every available fitted pipeline and score it on the same test observations.
4. Build a comparison table containing RMSE, MAE, and R2.
5. Rank models by ascending RMSE and select the first row as the best model.
6. Save machine-readable results to `reports/model_comparison.csv`.
7. Generate `reports/model_training_summary.md` for reviewers and CI artifacts.
8. Upload the selected artifact as `model.joblib` to `<HF_USERNAME>/superkart-best-sales-model`.

Using the same held-out test set makes the comparison consistent. The script tolerates missing individual artifacts, but at least one trained model must exist. In a production workflow, the test set should remain untouched until this final comparison stage, and registration should be protected by an explicit quality threshold.

In [17]:
model_files = {
    "Decision Tree": MODELS_DIR / "decision_tree_model.joblib",
    "Bagging": MODELS_DIR / "bagging_model.joblib",
    "Random Forest": MODELS_DIR / "random_forest_model.joblib",
    "AdaBoost": MODELS_DIR / "adaboost_model.joblib",
    "Gradient Boosting": MODELS_DIR / "gradient_boosting_model.joblib",
    "XGBoost": MODELS_DIR / "xgboost_model.joblib",
}

available_models = {
    model_name: model_path
    for model_name, model_path in model_files.items()
    if model_path.is_file()
}
if not available_models:
    raise FileNotFoundError(
        f"No trained model artifacts were found in {MODELS_DIR.resolve()}. "
        "Run at least one model-training cell before comparison."
    )

comparison_results = []
for model_name, model_path in available_models.items():
    fitted_model = joblib.load(model_path)
    predictions = fitted_model.predict(X_test)
    comparison_results.append(
        {"Model": model_name, **evaluate_regression(y_test, predictions)}
    )

comparison_df = pd.DataFrame(comparison_results)[["Model", "RMSE", "MAE", "R2"]]
comparison_df = comparison_df.sort_values("RMSE", ascending=True).reset_index(drop=True)
comparison_df.index = comparison_df.index + 1
comparison_df.index.name = "Rank"
display(comparison_df)

comparison_csv = REPORTS_DIR / "model_comparison.csv"
comparison_report = REPORTS_DIR / "model_training_summary.md"
comparison_df.to_csv(comparison_csv, index=True)

best_model_name = comparison_df.iloc[0]["Model"]
best_rmse = comparison_df.iloc[0]["RMSE"]
report_lines = [
    "# Automated Model Comparison",
    "",
    "| Rank | Model | RMSE | MAE | R2 |",
    "|---:|---|---:|---:|---:|",
]
for rank, row in comparison_df.reset_index().iterrows():
    report_lines.append(
        f"| {int(row['Rank'])} | {row['Model']} | {row['RMSE']:.6f} | "
        f"{row['MAE']:.6f} | {row['R2']:.6f} |"
    )
report_lines.extend(
    [
        "",
        "## Selected Model",
        "",
        f"**{best_model_name}** was selected because it achieved the lowest "
        f"RMSE ({best_rmse:.6f}).",
        "",
    ]
)
comparison_report.write_text("\n".join(report_lines), encoding="utf-8")

best_model_repo_id = f"{HF_USERNAME}/superkart-best-sales-model"
best_model_path = model_files[best_model_name]
api = HfApi(token=HF_TOKEN)
api.create_repo(
    repo_id=best_model_repo_id,
    repo_type="model",
    exist_ok=True,
)
api.upload_file(
    path_or_fileobj=str(best_model_path),
    path_in_repo="model.joblib",
    repo_id=best_model_repo_id,
    repo_type="model",
)

print(f"Comparison CSV: {comparison_csv.resolve()}")
print(f"Comparison report: {comparison_report.resolve()}")
print(f"Best model: {best_model_name} (RMSE={best_rmse:.6f})")
print(f"Registered at: https://huggingface.co/{best_model_repo_id}")

,Model,RMSE,MAE,R2
Rank,,,,
1,Random Forest,278.255953,110.315282,0.932143
2,Gradient Boosting,284.084412,121.345849,0.929270
3,Bagging,284.591376,111.297831,0.929018
4,XGBoost,285.994069,116.401680,0.928316
5,Decision Tree,332.465665,149.617112,0.903128
6,AdaBoost,449.957095,346.419979,0.822561


random_forest_model.joblib: 100%|██████████| 19.9M/19.9M [00:05<00:00, 3.96MB/s]


Comparison CSV: C:\Users\arvnarayanan\Documents\mlops_sales_forecasting\reports\model_comparison.csv
Comparison report: C:\Users\arvnarayanan\Documents\mlops_sales_forecasting\reports\model_training_summary.md
Best model: Random Forest (RMSE=278.255953)
Registered at: https://huggingface.co/arvind2221994/superkart-best-sales-model


## Deployment

The SuperKart Sales Revenue Predictor is deployed on **Streamlit Community Cloud**, while the fitted production model remains in **Hugging Face Model Hub**. Streamlit hosts the user interface and prediction service; Hugging Face is the source of truth for the versioned `model.joblib` artifact.

The deployed application is available at [SuperKart Sales Predictor](https://mlopssalesforecasting-sfvvmx7aozxtwfgjuwcv4f.streamlit.app/).

### Deployment Architecture

```text
User
  |
  v
Streamlit Community Cloud
  |  Runs deployment/app.py
  |  Downloads model.joblib
  v
Hugging Face Model Hub
<HF_USERNAME>/superkart-best-sales-model
```

The comparison workflow uploads the winning pipeline to `<HF_USERNAME>/superkart-best-sales-model`. When the Streamlit application starts, it constructs that repository ID, downloads `model.joblib`, deserializes the complete preprocessing-and-regression pipeline, caches it, and uses it to serve predictions.

### Deployment Files

| File | Purpose |
|---|---|
| `deployment/app.py` | Defines the Streamlit interface, downloads the model from Hugging Face, and produces predictions. |
| `deployment/requirements.txt` | Lists packages installed by Streamlit Community Cloud. |
| `deployment/deploy_to_hf.py` | Legacy Hugging Face Space deployment script; it is not used for the current deployment. |

### Why Streamlit Community Cloud Is Used

The original deployment attempted to use a Hugging Face Space with `space_sdk="streamlit"`. Hugging Face Spaces no longer supports Streamlit as a Space SDK; supported values are `gradio`, `docker`, and `static`. A static Space cannot run this Python model, and Docker Spaces require a paid plan. Streamlit Community Cloud can run the existing application directly without a Docker image or a Gradio rewrite.

### How Streamlit Connects to Hugging Face

The application reads the Hugging Face username from its environment and builds the production model repository ID:

```python
HF_USERNAME = os.getenv("HF_USERNAME", "your-hf-username")
MODEL_REPO_ID = f"{HF_USERNAME}/superkart-best-sales-model"
```

It downloads the registered artifact with `huggingface_hub`:

```python
model_path = hf_hub_download(
    repo_id=MODEL_REPO_ID,
    filename="model.joblib",
    repo_type="model",
)
```

The loader uses `@st.cache_resource`, so Streamlit does not download and deserialize the model again on every UI rerun. Because the current model repository is public, prediction deployment does not require `HF_TOKEN`. If the repository becomes private, add an authorized `HF_TOKEN` through Streamlit secrets—never in source code.

### Streamlit Community Cloud Deployment Steps

1. Confirm that the comparison workflow has registered `model.joblib` in `<HF_USERNAME>/superkart-best-sales-model`.
2. Confirm that `deployment/app.py` and `deployment/requirements.txt` are committed and pushed to GitHub.
3. Open [Streamlit Community Cloud](https://share.streamlit.io) and sign in with the GitHub account that can access the repository.
4. Create a new application and select the GitHub repository containing this project.
5. Select the `main` branch.
6. Set the main file path to `deployment/app.py`.
7. Select Python 3.12 to match the documented deployment environment.
8. Open **Advanced settings → Secrets** and add:

```toml
HF_USERNAME = "arvind2221994"
```

9. Deploy the application. Streamlit checks out the branch, installs `deployment/requirements.txt`, starts the server, reads `HF_USERNAME`, downloads the Hugging Face model, and renders the input form.
10. If the model is private, also add `HF_TOKEN` in Streamlit secrets and ensure the token has read access.

For serialization compatibility, pin `joblib`, NumPy, Pandas, scikit-learn, and XGBoost versions to versions compatible with the model-training environment.

### Deployment Verification

After deployment:

1. Open the public Streamlit URL.
2. Confirm that the SuperKart prediction form renders.
3. Confirm that **Model loaded successfully from Hugging Face Hub!** appears.
4. Enter representative product and store values.
5. Submit the form and confirm that a numeric predicted sales total is returned.
6. Review Streamlit build and runtime logs if startup or prediction fails.

The model-loaded message verifies that Streamlit can reach Hugging Face, resolve the configured repository, download the artifact, and deserialize the fitted pipeline.

### Updating and Troubleshooting the Deployment

Push application changes to the configured GitHub branch. Streamlit Community Cloud detects new commits and rebuilds or reruns the application automatically; `deployment/deploy_to_hf.py` is not part of this process.

| Symptom | Likely cause | Resolution |
|---|---|---|
| Repository resolves to `your-hf-username` | `HF_USERNAME` is missing. | Add it under Streamlit secrets and reboot the app. |
| Model download returns 401 or 403 | The model is private or access is missing. | Add an authorized `HF_TOKEN` through Streamlit secrets. |
| `ModuleNotFoundError` | A runtime dependency is missing. | Add it to `deployment/requirements.txt` and redeploy. |
| Model deserialization warns or fails | Training and deployment library versions differ. | Pin compatible versions of joblib, NumPy, Pandas, scikit-learn, and XGBoost. |
| Visitors are redirected to authentication | Public sharing is disabled or restricted. | Review sharing settings and test in a signed-out browser. |
| The app sleeps after inactivity | Normal Community Cloud behavior. | Reopen the URL and allow the application to restart. |

## Git repo and actions workflow setup

This section summarizes `GITHUB_SETUP.md`: first publish the project as a GitHub repository, then configure the existing GitHub Actions MLOps workflow. Do not place GitHub tokens, passwords, `.env` contents, or Hugging Face tokens directly in the notebook or repository.

### A. Create and Push the Git Repository

#### 1. Open the project folder

In Google Colab, change to the persistent project directory with `%cd`:

```python
%cd /content/mlops_sales_forecasting
```

Use the appropriate path when running somewhere else.

#### 2. Protect local and sensitive files

Ensure `.gitignore` excludes `.env`, local data, generated caches, MLflow data, and virtual environments including `.venv-x64/`. Verify sensitive paths before staging:

```bash
git check-ignore .env
git check-ignore .venv-x64/
```

#### 3. Initialize Git and name the primary branch

```bash
git init
git branch -M main
```

#### 4. Configure the commit author

```bash
git config user.name "YOUR NAME"
git config user.email "YOUR_GITHUB_EMAIL"
```

Use an email associated with the GitHub account or a GitHub-provided `noreply` address.

#### 5. Stage and review files

```bash
git add .
git status
```

Stop if `.env`, tokens, local data, virtual environments, or other sensitive files appear. Update `.gitignore`, unstage the path, and review again before committing.

#### 6. Create the initial commit

```bash
git commit -m "Initial commit: sales forecasting MLOps project"
```

#### 7. Install and authenticate GitHub CLI in Colab

```bash
sudo apt-get update -qq
sudo apt-get install -y gh
gh --version
gh auth login --hostname github.com --git-protocol https --web
gh auth status
```

The web login displays a one-time code and does not require storing a GitHub token in the notebook. Colab authentication is temporary and may need to be repeated in a new runtime.

#### 8. Create the GitHub repository and push

```bash
gh repo create mlops_sales_forecasting --public --source=. --remote=origin --push
```

This creates the repository, adds `origin`, and pushes `main`.

#### 9. Verify the remote and working tree

```bash
git remote -v
git branch -vv
git status
gh repo view
```

For later changes, stage and review the intended files, commit them with a descriptive message, and run `git push`.

### B. Configure the GitHub Actions Workflow

#### 1. Verify workflow discovery

GitHub discovers workflows under `.github/workflows`. Confirm that the pipeline exists at:

```text
.github/workflows/pipeline.yml
```

The workflow validates pull requests and performs training, model comparison, Hugging Face publication, artifact retention, and report generation for relevant updates to `main`.

#### 2. Configure repository variables

Open **Settings → Secrets and variables → Actions → Variables** and add:

| Variable | Value |
|---|---|
| `HF_USERNAME` | `arvind2221994` |
| `HF_DATASET_NAME` | `superkart-sales-forecasting` |

These values identify public Hugging Face resources and are not credentials.

#### 3. Configure the Hugging Face secret

Under **Settings → Secrets and variables → Actions → Secrets**, create:

| Secret | Value |
|---|---|
| `HF_TOKEN` | A Hugging Face token with permission to update the dataset and model repositories |

Never place this token in `pipeline.yml`, `.env.example`, documentation, source files, or notebook output. GitHub injects it only into the workflow runtime.

#### 4. Allow report commits

Open **Settings → Actions → General → Workflow permissions**, select **Read and write permissions**, and save. The workflow needs this permission to commit generated comparison reports. If branch protection blocks direct pushes, allow the workflow identity or change the reporting step to create a pull request.

#### 5. Commit and push the workflow files

```bash
git add .github/workflows/pipeline.yml
git add scripts/compare_models.py
git add GITHUB_SETUP.md
git status
git commit -m "Add automated MLOps training workflow"
git push origin main
```

Open the repository's **Actions** tab and confirm that **MLOps Training Pipeline** appears.

#### 6. Run the MLOps pipeline

The workflow runs automatically for configured changes pushed to `main`; pull requests run validation only. To run it manually:

1. Open the GitHub repository's **Actions** tab.
2. Select **MLOps Training Pipeline**.
3. Select **Run workflow**.
4. Choose the `main` branch.
5. Select **Run workflow** again.

The training job consumes the prepared Hugging Face `train` and `test` splits. It intentionally does not run `prepare_data.py`, because repeatedly splitting the existing `train` split would change and progressively shrink the data.

#### 7. Review workflow outputs

After a successful run:

- Review the workflow logs.
- Download the `model-training-<run-number>` artifact containing generated models and reports.
- Confirm that all individual model repositories and `superkart-best-sales-model` were updated in Hugging Face.
- Review `reports/model_comparison.csv`.
- Review `reports/model_training_summary.md`.
- Confirm that the generated report commit reached `main`.

If configuration validation fails, confirm that `HF_TOKEN`, `HF_USERNAME`, and `HF_DATASET_NAME` use these exact names and that the token has the required Hugging Face write access.

## Final Links

| Asset | Link |
|---|---|
| Git repository | [arvind2221994/mlops_sales_forecasting](https://github.com/arvind2221994/mlops_sales_forecasting/) |
| GitHub Actions workflow | [Workflow runs - arvind2221994/mlops_sales_forecasting](https://github.com/arvind2221994/mlops_sales_forecasting/actions) |
| Streamlit application | [SuperKart Sales Predictor - Streamlit](https://mlopssalesforecasting-sfvvmx7aozxtwfgjuwcv4f.streamlit.app/) |